# Notebook mẫu: huấn luyện và đánh giá một mô hình (E1, E2)

**Cách dùng**
1. Copy file này thành `04_models_<tên bạn>.ipynb` (không sửa file mẫu).
2. Đổi `AUTHOR` và `MODEL_NAME` ở cell đầu.
3. Thay mô hình ở cell *Mô hình của bạn* (mẫu đang dùng Logistic Regression). Phần còn lại giữ nguyên.
4. Chạy *Restart & Run All*. Kết quả tự ghi vào `reports/results/<AUTHOR>.csv` và xác suất vào `models/`.

**Quy tắc**
- Chỉ dùng `train` và `val`. Không mở `test` (hàm đọc sẽ báo lỗi).
- Mọi bước học số liệu nằm trong `Pipeline`, fit trên train.
- Chọn ngưỡng và tinh chỉnh tham số trên `val`, không bao giờ trên `test`.
- Nếu AUC hoặc F1 cao bất thường (ví dụ trên 0,95), nghi ngờ rò rỉ dữ liệu trước khi vui mừng.

In [ ]:
import sys; sys.path.insert(0, "..")   # để import được thư mục src/

import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

from src.data import load_xy, SEED
from src.features import build_preprocessor, check_columns
from src.evaluate import evaluate, evaluate_by_group, log_result, save_proba

AUTHOR = "ten_ban"          # đổi thành tên bạn, viết thường không dấu, ví dụ "an"
MODEL_NAME = "logreg"       # tên ngắn của mô hình, dùng đặt tên file xác suất

## Đọc dữ liệu

In [ ]:
X_train, y_train = load_xy("train")
X_val, y_val = load_xy("val")
check_columns(X_train)   # báo lỗi nếu có cột lạ hoặc thiếu cột
print(f"train {X_train.shape}, tỷ lệ hủy {y_train.mean():.1%}")
print(f"val   {X_val.shape}, tỷ lệ hủy {y_val.mean():.1%}")

## Baseline

Luôn bắt đầu từ mốc này. Mô hình đoán "không hủy" cho mọi đơn sẽ cho accuracy cao nhưng recall bằng 0: đó là lý do không dùng accuracy một mình.

In [ ]:
dummy = DummyClassifier(strategy="prior").fit(X_train, y_train)
m_dummy = evaluate(y_val, dummy.predict_proba(X_val)[:, 1])
log_result(AUTHOR, "E1", "dummy", "val", m_dummy)
pd.Series(m_dummy)

## Mô hình của bạn (E1)

**Chỗ cần sửa:** thay `LogisticRegression(...)` bằng mô hình của bạn và chọn `kind`:
- `"linear"` cho Logistic Regression và MLP (cần chuẩn hóa, one-hot)
- `"tree"` cho Decision Tree, Random Forest, XGBoost, LightGBM

In [ ]:
model = Pipeline([
    ("prep", build_preprocessor("linear")),
    ("clf", LogisticRegression(max_iter=1000, random_state=SEED)),
])
model.fit(X_train, y_train)

proba_val = model.predict_proba(X_val)[:, 1]
m = evaluate(y_val, proba_val, threshold=0.5)
log_result(AUTHOR, "E1", MODEL_NAME, "val", m)
save_proba(MODEL_NAME, proba_val, split="val")
pd.Series(m)

Chỉ số tách theo khách sạn (yêu cầu báo cáo theo nhóm, không chỉ con số tổng):

In [ ]:
evaluate_by_group(y_val, proba_val, X_val["hotel"]).round(3)

## E2: xử lý mất cân bằng lớp

Thử `class_weight="balanced"` trước (một tham số, không thêm dữ liệu, không rủi ro rò rỉ). SMOTE nếu dùng thì chỉ áp dụng trong Pipeline của `imblearn`, trên tập huấn luyện.

In [ ]:
model_bal = Pipeline([
    ("prep", build_preprocessor("linear")),
    ("clf", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=SEED)),
]).fit(X_train, y_train)

proba_bal = model_bal.predict_proba(X_val)[:, 1]
m_bal = evaluate(y_val, proba_bal, threshold=0.5)
log_result(AUTHOR, "E2", MODEL_NAME, "val", m_bal, note="class_weight=balanced")
save_proba(MODEL_NAME + "_balanced", proba_bal, split="val")

## So sánh

In [ ]:
cols = ["precision", "recall", "f1", "roc_auc", "pr_auc", "accuracy"]
pd.DataFrame({"baseline": m_dummy, MODEL_NAME: m, MODEL_NAME + " + balanced": m_bal}).T[cols].round(3)

## Việc tiếp theo của bạn
- Tinh chỉnh tham số trên `val` (ví dụ `GridSearchCV` hoặc `cross_val_score` trong tập train, đưa **cả Pipeline** vào).
- Chọn ngưỡng theo mục tiêu (nhiều recall hay nhiều precision) trên `val`, ghi lại lý do.
- Mỗi lần có kết quả mới, gọi `log_result` và `save_proba` như trên.
- Ghi vào báo cáo: vì sao chọn tham số này, mô hình sai ở đâu (tách theo nhóm), so với baseline được bao nhiêu.